In [7]:
K = [0xd76aa478, 0xe8c7b756, 0x242070db, 0xc1bdceee, 
    0xf57c0faf, 0x4787c62a, 0xa8304613, 0xfd469501,
    0x698098d8, 0x8b44f7af, 0xffff5bb1, 0x895cd7be,
    0x6b901122, 0xfd987193, 0xa679438e, 0x49b40821,
    0xf61e2562, 0xc040b340, 0x265e5a51, 0xe9b6c7aa,
    0xd62f105d, 0x02441453, 0xd8a1e681, 0xe7d3fbc8,
    0x21e1cde6, 0xc33707d6, 0xf4d50d87, 0x455a14ed,
    0xa9e3e905, 0xfcefa3f8, 0x676f02d9, 0x8d2a4c8a,
    0xfffa3942, 0x8771f681, 0x6d9d6122, 0xfde5380c,
    0xa4beea44, 0x4bdecfa9, 0xf6bb4b60, 0xbebfbc70,
    0x289b7ec6, 0xeaa127fa, 0xd4ef3085, 0x04881d05,
    0xd9d4d039, 0xe6db99e5, 0x1fa27cf8, 0xc4ac5665,
    0xf4292244, 0x432aff97, 0xab9423a7, 0xfc93a039,
    0x655b59c3, 0x8f0ccc92, 0xffeff47d, 0x85845dd1,
    0x6fa87e4f, 0xfe2ce6e0, 0xa3014314, 0x4e0811a1,
    0xf7537e82, 0xbd3af235, 0x2ad7d2bb, 0xeb86d391]

S = [7, 12, 17, 22,  7, 12, 17, 22,  7, 12, 17, 22,  7, 12, 17, 22,
    5,  9, 14, 20,  5,  9, 14, 20,  5,  9, 14, 20,  5,  9, 14, 20,
    4, 11, 16, 23,  4, 11, 16, 23,  4, 11, 16, 23,  4, 11, 16, 23,
    6, 10, 15, 21,  6, 10, 15, 21,  6, 10, 15, 21,  6, 10, 15, 21]

In [8]:
def padding(m: bytes):
    ms = len(m) * 8

    k = (56 - (len(m) + 1)) % 64

    m += b'\x80'
    m += b'\x00' * k
    m += ms.to_bytes(8, byteorder='little')
    return m

In [9]:
def combine(A,B,C,D,i,M):
    if i < 16:
        F = (B & C) | ((~B) & D)
        g = i
    elif i < 32:
        F = (B & D) | (C & (~D))
        g = (5*i + 1) % 16
    elif i < 48:
        F = B ^ C ^ D
        g = (3*i + 5) % 16
    else:
        F = C ^ (B | (~D))
        g = (7*i) % 16

    F = (F + A + K[i] + M[g]) & 0xFFFFFFFF
    F = ((F << S[i]) | (F >> (32 - S[i]))) & 0xFFFFFFFF
    F = (F + B) & 0xFFFFFFFF
    return F
    

In [19]:
def hash(m: bytes):
    m = padding(m)
    
    A0 = 0x67452301
    B0 = 0xefcdab89
    C0 = 0x98badcfe
    D0 = 0x10325476

    for i in range(0, len(m), 64):
        block = m[i:i+64]
        M = [int.from_bytes(block[j:j+4], byteorder='little') for j in range(0,64,4)]
        A,B,C,D = A0,B0,C0,D0

        for j in range(64):
            F = combine(A,B,C,D,j,M)
            A,B,C,D = D,F,B,C
        A0 = (A0 + A) & 0xFFFFFFFF
        B0 = (B0 + B) & 0xFFFFFFFF
        C0 = (C0 + C) & 0xFFFFFFFF
        D0 = (D0 + D) & 0xFFFFFFFF
    
    return (A0.to_bytes(4,'little') + 
            B0.to_bytes(4,'little') + 
            C0.to_bytes(4,'little') + 
            D0.to_bytes(4,'little'))

In [20]:
msg = input()
msg_b = msg.encode('utf-8')
print(hash(msg_b).hex())

0cc175b9c0f1b6a831c399e269772661
